In [ ]:
# ── Core libraries ───────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ── Sklearn: data splitting & scaling ────────────────────────────────
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder

# ── Sklearn: models ──────────────────────────────────────────────────
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA

# ── Sklearn: metrics ─────────────────────────────────────────────────
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
    accuracy_score
)

# ── XGBoost ──────────────────────────────────────────────────────────
from xgboost import XGBClassifier

In [ ]:
df = pd.read_csv("final-Project.csv")

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
df['Number of Victims / Nombre de victimes'] = df['Number of Victims / Nombre de victimes'].fillna(0).astype(int)
df['Dollar Loss /pertes financières'] = df['Dollar Loss /pertes financières'].replace({r'[$,]': ''}, regex=True).astype(float)

df

In [ ]:
df.info()

In [ ]:
df.columns

In [ ]:
df = df.rename(columns={
    "Numéro d'identification / Number ID": "Number ID",
    "Date Received / Date reçue": "Date Received",
    "Victim Age Range / Tranche d'âge des victimes" : "Victim Age Range",
    "Number of Victims / Nombre de victimes":"Number of Victims",
    "Dollar Loss /pertes financières": "Dollar Loss"
})
display(df.head())

In [ ]:
df["Log_Dollar_Loss"] = np.log1p(df["Dollar Loss"])
df = df.drop(columns=["Dollar Loss"])

# should i drop pays? if not please let me know what should i do ?

In [ ]:
columns_to_drop = [
    'Pays',
    'Genre',
    'Type de plainte',
    'Province/Etat',
    'Categories thematiques sur la fraude et la cybercriminalite',
    'Langue de correspondance','Methode de sollicitation','Type de plainte recue', 'Catégories thématiques sur la fraude et la cybercriminalité','Méthode de sollicitation','Province/État','Type de plainte reçue'
]

existing_columns_to_drop = [col for col in columns_to_drop if col in df.columns]
df = df.drop(columns=existing_columns_to_drop)

display(df.columns)

In [ ]:
df['Victim Age Range'] = df['Victim Age Range'].str.lstrip("' ")

In [ ]:
df

In [ ]:
df['Date Received'] = pd.to_datetime(df['Date Received'], errors='coerce')
df['Month']      = df['Date Received'].dt.month
df['MonthName']  = df['Date Received'].dt.month_name().str[:3]
df['DayOfWeek']  = df['Date Received'].dt.dayofweek
df['DayName']    = df['Date Received'].dt.day_name()

In [ ]:
month_grp = df.groupby(['Month','MonthName']).agg(
    Cases=('Month','count'),
    Total_Loss=('Log_Dollar_Loss','sum')
).reset_index().sort_values('Month')

day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
day_grp = df.groupby('DayName').agg(
    Cases=('DayName','count'),
    Total_Loss=('Log_Dollar_Loss','sum')
).reindex(day_order).reset_index()


In [ ]:
import matplotlib.gridspec as gridspec
import matplotlib.pyplot as plt

# Define colors and bar_colors function
BASE_BLUE = '#264653'
BASE_TEAL = '#2A9D8F'

def bar_colors(data, base_color):
    norm = plt.Normalize(data.min(), data.max())
    cmap = plt.cm.get_cmap('YlGnBu')
    return cmap(norm(data))


fig = plt.figure(figsize=(14, 10))
fig.suptitle('Fraud Reports — Cases & Dollar Loss by Date', fontsize=14, fontweight='bold', y=1.01)
gs  = gridspec.GridSpec(2, 2, hspace=0.55, wspace=0.35)

# Monthly cases
ax1 = fig.add_subplot(gs[0, 0])
ax1.bar(month_grp['MonthName'], month_grp['Cases'],
        color=bar_colors(month_grp['Cases'], BASE_BLUE), edgecolor='none')
ax1.set_title('Cases by Month', fontsize=12, fontweight='bold')
ax1.set_ylabel('Number of Cases')
ax1.tick_params(axis='x', rotation=45)
for i, v in enumerate(month_grp['Cases']):
    ax1.text(i, v + 200, f'{v:,}', ha='center', fontsize=7.5)

# Monthly loss
ax2 = fig.add_subplot(gs[0, 1])
loss_m = month_grp['Total_Loss'] / 1e6
ax2.bar(month_grp['MonthName'], loss_m,
        color=bar_colors(loss_m, BASE_TEAL), edgecolor='none')
ax2.set_title('Dollar Loss by Month ($M)', fontsize=12, fontweight='bold')
ax2.set_ylabel('Total Loss ($M)')
ax2.tick_params(axis='x', rotation=45)
for i, v in enumerate(loss_m):
    ax2.text(i, v + 1, f'${v:.0f}M', ha='center', fontsize=7.5)

# Daily cases
ax3 = fig.add_subplot(gs[1, 0])
day_labels = [d[:3] for d in day_grp['DayName']]
ax3.bar(day_labels, day_grp['Cases'],
        color=bar_colors(day_grp['Cases'], BASE_BLUE), edgecolor='none')
ax3.set_title('Cases by Day of Week', fontsize=12, fontweight='bold')
ax3.set_ylabel('Number of Cases')
for i, v in enumerate(day_grp['Cases']):
    ax3.text(i, v + 200, f'{v:,}', ha='center', fontsize=7.5)

# Daily loss
ax4 = fig.add_subplot(gs[1, 1])
loss_d = day_grp['Total_Loss'] / 1e6
ax4.bar(day_labels, loss_d,
        color=bar_colors(loss_d, BASE_TEAL), edgecolor='none')
ax4.set_title('Dollar Loss by Day of Week ($M)', fontsize=12, fontweight='bold')
ax4.set_ylabel('Total Loss ($M)')
for i, v in enumerate(loss_d):
    ax4.text(i, v + 2, f'${v:.0f}M', ha='center', fontsize=7.5)

for ax in [ax1,ax2,ax3,ax4]:
    ax.spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.show()

# ── NUMBER TABLE ───────────────────────────────────────
print('\n=== By Month ===')
month_grp['Avg_Loss'] = (month_grp['Total_Loss'] / month_grp['Cases']).round(0)
month_grp['Loss_M']   = (month_grp['Total_Loss'] / 1e6).round(1)
month_grp['Pct']      = (month_grp['Cases'] / month_grp['Cases'].sum() * 100).round(1)
print(month_grp[['MonthName','Cases','Pct','Loss_M','Avg_Loss']].to_string(index=False))

print('\n=== By Day of Week ===')
day_grp['Avg_Loss'] = (day_grp['Total_Loss'] / day_grp['Cases']).round(0)
day_grp['Loss_M']   = (day_grp['Total_Loss'] / 1e6).round(1)
day_grp['Pct']      = (day_grp['Cases'] / day_grp['Cases'].sum() * 100).round(1)
print(day_grp[['DayName','Cases','Pct','Loss_M','Avg_Loss']].to_string(index=False))

In [ ]:
df

In [ ]:
columns_to_drop = ['Date Received', 'Year', 'DayName','MonthName']
df = df.drop(columns=columns_to_drop, errors='ignore')

# Display the first few rows to confirm the changes
display(df.head())

# work on date and keep it in your data set!

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
# df = df.drop(columns=['_id', 'Number ID', 'Date Received'])

# print(df.shape)
# df.head()

In [ ]:

print('===== 1. Log Dollar Loss Distribution =====')
plt.figure(figsize=(10,4))
df['Log_Dollar_Loss'].dropna().hist(bins=50)
plt.title('Distribution of Log Dollar Loss')
plt.xlabel('Log Dollar Loss')
plt.show()

print('===== 2. Top Provinces =====')
df['Province/State'].value_counts().head(10).plot(kind='bar', figsize=(10,4), color='steelblue')
plt.title('Cases by Province/State')
plt.tight_layout()
plt.show()

print('===== 3. Victim Age Range =====')
df['Victim Age Range'].value_counts().plot(kind='bar', figsize=(10,4), color='coral')
plt.title('Victim Age Range Distribution')
plt.tight_layout()
plt.show()

print('===== 4. Solicitation Method =====')
if 'Solicitation Method' in df.columns:
    df['Solicitation Method'].value_counts().plot(kind='bar', figsize=(10,4), color='mediumseagreen')
    plt.title('Solicitation Method')
    plt.tight_layout()
    plt.show()

print('===== 5. Complaint Type =====')
if 'Complaint Type' in df.columns:
    df['Complaint Type'].value_counts().head(10).plot(kind='bar', figsize=(12,4), color='mediumpurple')
    plt.title('Top 10 Complaint Types')
    plt.tight_layout()
    plt.show()

print('===== 6. Log Dollar Loss by Age Range (Boxplot) =====')
plt.figure(figsize=(12,5))
df.dropna(subset=['Victim Age Range', 'Log_Dollar_Loss']).boxplot(
    column='Log_Dollar_Loss', by='Victim Age Range', figsize=(12,5)
)
plt.title('Log Dollar Loss by Age Range')
plt.suptitle('')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Scatter diagram

In [ ]:
print('===== 7. Full Correlation Heatmap')
df_encoded = df.copy()
cat_cols = df_encoded.select_dtypes(include='object').columns

for col in cat_cols:
    df_encoded[col] = pd.factorize(df_encoded[col])[0]
# just numeric columns
numeric_df = df_encoded.select_dtypes(include='number')

plt.figure(figsize=(12,8))
sns.heatmap(numeric_df.corr(), annot=True, fmt='.2f', cmap='coolwarm', linewidths=0.5)
plt.title('Correlation Heatmap - All Features')
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(12, 6))
sns.boxplot(data=df, x='Victim Age Range', y='Log_Dollar_Loss')
plt.title('Distribution of Log Dollar Loss by Victim Age Range')
plt.xlabel('Victim Age Range')
plt.ylabel('Log Dollar Loss')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
print("data set beore encoding:", df.shape)

In [ ]:
df_encoded = df.copy()
columns_to_drop = ['Number ID']
# Check if '_id' exists in df.columns before trying to drop it
if '_id' in df_encoded.columns:
    columns_to_drop.append('_id')

df_encoded = df_encoded.drop(columns=columns_to_drop)
print("encoded data set:", df_encoded.shape)

In [ ]:
def categorize_age(age):
    if pd.isna(age):
        return None
    age_str = str(age).strip()

    if age_str in ["Under 18", "1 - 9", "10 - 19"]:
        return "Young"
    elif age_str in ["20 - 29", "30 - 39", "40 - 49", "50 - 59"]:
        return "Adult"
    elif age_str in ["60 - 69", "70 - 79", "80 - 89", "90 - 99", "100 +"]:
        return "Senior"
    elif age_str in ["Deceased / Décédé", "Business / Entreprise",
                     "Not Available / non disponible"]:
        return None
    else:
        return None

# Apply categorization
df_encoded["Age_Group"] = df_encoded["Victim Age Range"].apply(categorize_age)

# Drop rows where Age_Group is None
df_encoded = df_encoded.dropna(subset=["Age_Group"]).reset_index(drop=True)

print("After creating Age_Group → Shape:", df_encoded.shape)
print("\nAge Group Distribution:")
print(df_encoded["Age_Group"].value_counts())

# Drop original Victim Age Range (optional but recommended)
df_encoded = df_encoded.drop(columns=["Victim Age Range"], errors='ignore')

In [ ]:


# Better version (recommended) for One-Hot Encoding
categorical_cols = [
    'Country',
    'Province/State',
    'Fraud and Cybercrime Thematic Categories',
    'Solicitation Method',
    'Gender',
    'Language of Correspondence',
    'Complaint Received Type',
    'Complaint Type'
]

df_encoded = pd.get_dummies(df_encoded,
                            columns=categorical_cols,
                            drop_first=True,
                            dtype='int8')   # saves memory

print("After One-Hot Encoding → Shape:", df_encoded.shape)
print("Dtypes:\n", df_encoded.dtypes.value_counts())
df_encoded.head()

# Classification


# What is the Bussiness Q?

# Add **how much dollar loss for each  age class**! search if we can predict dollar loss for each group!

# Look to the **country**

In [ ]:
# X and y
y = df_encoded["Age_Group"]
X = df_encoded.drop(columns=["Age_Group"])

# Train / Val / Test split
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42
)



In [ ]:
# Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled   = scaler.transform(X_val)
X_test_scaled  = scaler.transform(X_test)

print("Train shape:", X_train.shape)

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

# Compute weights — Young gets ~7.8x more weight than Adult
weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)
class_weights = dict(zip(np.unique(y_train), weights))
print("Class weights:", class_weights)

In [ ]:
# ── Define all pipelines ────────────────────────────────────────────
# Pipeline ensures: scaler is fit ONLY on train, never on val/test
# No more X_train_scaled / X_val_scaled needed — pipeline handles it internally

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from xgboost import XGBClassifier


# ── LabelEncoder for XGBoost (needs numeric labels) ──────────────────
le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_val_enc   = le.transform(y_val)
y_test_enc  = le.transform(y_test)
print("Label mapping:", dict(zip(le.classes_, le.transform(le.classes_))))

# ── Logistic Regression ───────────────────────────────────────────────
pipe_lr = Pipeline([
    ('scaler', StandardScaler()),
    ('clf',    LogisticRegression(
                   C=1,
                   max_iter=500,
                   class_weight=class_weights,   # ← computed weights
                   random_state=42))
])

# ── Decision Tree ─────────────────────────────────────────────────────
pipe_dt = Pipeline([
    ('clf', DecisionTreeClassifier(
                max_depth=10,
                class_weight=class_weights,       # ← computed weights
                random_state=42))
])

# ── Random Forest ─────────────────────────────────────────────────────
pipe_rf = Pipeline([
    ('clf', RandomForestClassifier(
                n_estimators=300,
                max_depth=15,
                class_weight=class_weights,       # ← computed weights
                n_jobs=-1,
                random_state=42))
])

# ── XGBoost ───────────────────────────────────────────────────────────
# XGBoost does not support class_weight dict directly
# Use scale_pos_weight only for binary — for multiclass use sample_weight
from sklearn.utils.class_weight import compute_sample_weight

sample_weights = compute_sample_weight(
    class_weight='balanced',
    y=y_train
)

pipe_xgb = Pipeline([
    ('clf', XGBClassifier(
                n_estimators=300,
                max_depth=6,
                learning_rate=0.1,
                subsample=0.8,
                colsample_bytree=0.8,
                eval_metric='mlogloss',
                n_jobs=-1,
                random_state=42))
])

# ── SVM Linear ────────────────────────────────────────────────────────
pipe_svm = Pipeline([
    ('scaler', StandardScaler()),
    ('clf',    LinearSVC(
                   C=1.0,
                   max_iter=2000,
                   class_weight=class_weights,   # ← computed weights
                   random_state=42))
])

print("All pipelines defined!")

In [ ]:
# ── Train all pipelines ───────────────────────────────────────────────
import time
train_times = {}

t0 = time.time()
pipe_lr.fit(X_train, y_train)
train_times['Logistic Regression'] = round(time.time() - t0, 2)
print(f"LR      done → {train_times['Logistic Regression']}s")

t0 = time.time()
pipe_dt.fit(X_train, y_train)
train_times['Decision Tree'] = round(time.time() - t0, 2)
print(f"DT      done → {train_times['Decision Tree']}s")

t0 = time.time()
pipe_rf.fit(X_train, y_train)
train_times['Random Forest'] = round(time.time() - t0, 2)
print(f"RF      done → {train_times['Random Forest']}s")

t0 = time.time()
pipe_xgb.fit(X_train, y_train_enc,
             clf__sample_weight=sample_weights)
train_times['XGBoost'] = round(time.time() - t0, 2)
print(f"XGBoost done → {train_times['XGBoost']}s")

t0 = time.time()
pipe_svm.fit(X_train, y_train)
train_times['SVM Linear'] = round(time.time() - t0, 2)
print(f"SVM     done → {train_times['SVM Linear']}s")

print("\nAll models trained!")

In [ ]:


# Class names — must match y_train values
CLASSES = sorted(y_train.unique())   # ['Adult', 'Senior', 'Young']

def evaluate_model(pipe, X_val, X_test, name, encoded=False):
    """
    pipe    : trained sklearn Pipeline
    X_val   : validation features (unscaled — pipeline handles scaling)
    X_test  : test features (unscaled — pipeline handles scaling)
    name    : model name for chart title
    encoded : True only for XGBoost (outputs numeric labels → convert back)
    """

    # ── Get predictions ─────────────────────────────────────────────
    if encoded:
        # XGBoost predicts 0/1/2 → convert back to 'Adult'/'Senior'/'Young'
        yv = le.inverse_transform(pipe.predict(X_val))
        yt = le.inverse_transform(pipe.predict(X_test))
    else:
        yv = pipe.predict(X_val)
        yt = pipe.predict(X_test)

    # ── Compute summary metrics ──────────────────────────────────────
    val_f1   = f1_score(y_val,  yv, average='macro')
    test_f1  = f1_score(y_test, yt, average='macro')
    val_acc  = accuracy_score(y_val,  yv)
    test_acc = accuracy_score(y_test, yt)

    # ── Build figure with 3 panels ───────────────────────────────────
    fig = plt.figure(figsize=(16, 5))
    fig.suptitle(name, fontsize=14, fontweight='bold')
    gs = gridspec.GridSpec(1, 3, wspace=0.4)

    # Panel 1 — Confusion Matrix (test set only)
    ax1 = fig.add_subplot(gs[0])
    cm = confusion_matrix(y_test, yt, labels=CLASSES)
    ConfusionMatrixDisplay(cm, display_labels=CLASSES).plot(
        ax=ax1, colorbar=False, cmap='Blues')
    ax1.set_title('Confusion Matrix (Test)', fontweight='bold')

    # Panel 2 — Per-class F1: validation vs test side by side
    ax2 = fig.add_subplot(gs[1])
    rv = classification_report(y_val,  yv, output_dict=True)
    rt = classification_report(y_test, yt, output_dict=True)
    f1v = [rv[c]['f1-score'] for c in CLASSES]   # val F1 per class
    f1t = [rt[c]['f1-score'] for c in CLASSES]   # test F1 per class
    x = np.arange(len(CLASSES)); w = 0.35
    ax2.bar(x - w/2, f1v, w, label='Validation', color='#378ADD', alpha=0.85)
    ax2.bar(x + w/2, f1t, w, label='Test',       color='#1D9E75', alpha=0.85)
    ax2.set_xticks(x); ax2.set_xticklabels(CLASSES)
    ax2.set_ylim(0, 1.12)
    ax2.set_ylabel('F1 Score')
    ax2.set_title('Per-class F1', fontweight='bold')
    ax2.legend(fontsize=9)
    ax2.spines[['top', 'right']].set_visible(False)
    for i, (v, t) in enumerate(zip(f1v, f1t)):
        ax2.text(i - w/2, v + 0.02, f'{v:.2f}', ha='center', fontsize=8)
        ax2.text(i + w/2, t + 0.02, f'{t:.2f}', ha='center', fontsize=8)

    # Panel 3 — Scorecard table
    ax3 = fig.add_subplot(gs[2])
    ax3.axis('off')
    rows = [
        ['Metric',    'Val',           'Test'],
        ['Accuracy',  f'{val_acc:.3f}',  f'{test_acc:.3f}'],
        ['F1 Macro',  f'{val_f1:.3f}',   f'{test_f1:.3f}'],
        *[[f'F1 {c}',
           f'{rv[c]["f1-score"]:.3f}',
           f'{rt[c]["f1-score"]:.3f}'] for c in CLASSES]
    ]
    tbl = ax3.table(cellText=rows[1:], colLabels=rows[0],
                    cellLoc='center', loc='center')
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(10)
    tbl.scale(1, 1.6)
    for (r, c), cell in tbl.get_celld().items():
        if r == 0:
            cell.set_facecolor('#E6F1FB')
            cell.set_text_props(fontweight='bold')
        cell.set_edgecolor('#cccccc')
    ax3.set_title('Scorecard', fontweight='bold', pad=10)

    plt.tight_layout()
    plt.show()

    # ── Print text report ────────────────────────────────────────────
    print(f'\nVal F1={val_f1:.3f} | Test F1={test_f1:.3f}')
    print(classification_report(y_test, yt))

    # ── Return metrics dict for final comparison ─────────────────────
    return {
        'name':     name,
        'val_f1':   val_f1,
        'test_f1':  test_f1,
        'val_acc':  val_acc,
        'test_acc': test_acc
    }

print("evaluate_model() ready!")

In [ ]:
results = []

results.append(evaluate_model(
    pipe_lr,  X_val, X_test, 'Logistic Regression'))

results.append(evaluate_model(
    pipe_dt,  X_val, X_test, 'Decision Tree'))

results.append(evaluate_model(
    pipe_rf,  X_val, X_test, 'Random Forest'))

results.append(evaluate_model(
    pipe_xgb, X_val, X_test, 'XGBoost', encoded=True))  # encoded=True → uses le.inverse_transform

results.append(evaluate_model(
    pipe_svm, X_val, X_test, 'SVM Linear'))

# ── Final comparison table ───────────────────────────────────────────
import pandas as pd

comp = pd.DataFrame(results)[['name','val_f1','test_f1','val_acc','test_acc']]
comp.columns = ['Model', 'Val F1', 'Test F1', 'Val Acc', 'Test Acc']
comp = comp.sort_values('Test F1', ascending=False).reset_index(drop=True)

print('=' * 58)
print('FINAL MODEL COMPARISON')
print('=' * 58)
print(comp.to_string(index=False, float_format='{:.3f}'.format))
print('=' * 58)
print(f'Best model: {comp.iloc[0]["Model"]}')
print(f'Test F1   : {comp.iloc[0]["Test F1"]:.3f}')

# ── Final comparison bar chart ───────────────────────────────────────
fig, ax = plt.subplots(figsize=(11, 5))
x = np.arange(len(comp))
w = 0.3

b1 = ax.bar(x - w/2, comp['Val F1'],  w,
            label='Val F1',  color='#378ADD', alpha=0.85)
b2 = ax.bar(x + w/2, comp['Test F1'], w,
            label='Test F1', color='#1D9E75', alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels(comp['Model'], rotation=15, ha='right')
ax.set_ylim(0, 1.12)
ax.set_ylabel('F1 Macro')
ax.set_title('Final Model Comparison — Val vs Test F1',
             fontsize=13, fontweight='bold')
ax.legend()
ax.spines[['top', 'right']].set_visible(False)

# Add value labels on top of each bar
for bar in list(b1) + list(b2):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.01,
            f'{bar.get_height():.3f}',
            ha='center', fontsize=9)

# Highlight the best model with a gold background
ax.axvspan(x[0] - 0.45, x[0] + 0.45, alpha=0.08, color='gold')
ax.text(x[0], 1.07, 'BEST', ha='center',
        fontsize=9, color='#BA7517', fontweight='bold')

plt.tight_layout()
plt.show()

Final Comments:
If we do these 5 things we will significantly increase our accuracy and grade:

1. Add 'Loss_per_Victim' feature
2. Add time features (Month/Year)
3. Tune Random Forest (GridSearch)
4. Tune Logistic Regression (C parameter)
5. Reduce noisy features, as much as we can

Thanks
Lazaro